# 00 — Setup and data audit

No API calls. Mirrors the public, read-only parts of `sksoumik/judging-the-judges` into `data/upstream/` and loads the aggregated metrics cache.

**Confirmed limitation (do not re-derive, just carry forward):** `results/raw/` (per-instance verdicts) and `data/human_study/` are `.gitignore`'d in the upstream repo and were never public. Only `results/aggregated/*_metrics.json` (summary counts) exists, so downstream notebooks work from aggregated data only.

In [ ]:
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

subprocess.run(["bash", "scripts/fetch_upstream.sh"], cwd=PROJECT_ROOT, check=True)

In [ ]:
from src.data_audit import load_aggregated_metrics, reconcile_table1

UPSTREAM = PROJECT_ROOT / "data" / "upstream"
aggregated = load_aggregated_metrics(UPSTREAM)
print(f"{len(aggregated)} aggregated metrics files loaded")
print("models found:", sorted(aggregated['model'].unique()))
print("benchmarks found:", sorted(aggregated['benchmark'].unique()))
aggregated.head()

In [ ]:
# Acceptance check: aggregated cache must reconcile with the published Table 1
# (MT-Bench) within 0.5 points before anything downstream is trusted.
table1_csv = UPSTREAM / "analysis" / "figures" / "table1_main_results.csv"
report = reconcile_table1(aggregated, table1_csv, tolerance=0.005)
assert len(report) == 25, f"expected 25 reconciled cells, got {len(report)}"
assert report["within_tolerance"].all(), report[~report["within_tolerance"]]
print("PASS: all 25 model x strategy cells match Table 1 within 0.005")
report